In [1]:
import json
import yaml
import requests
import time
from pprint import pprint
from mstrio.connection import Connection
from mstr_robotics.mstr_classes import mstr_global, md_searches
from mstr_robotics.redis_db import  redis_bi_analysis,redis_mstr_json
from mstr_robotics._connectors import mstr_api
from mstr_robotics._helper import msic
from mstr_robotics.read_out_prj_obj import read_gen
from mstr_robotics.prepare_AI_data import export_mstr_md
from mstrio.api import user_hierarchies
import os
import re
from openai import OpenAI
from dotenv import load_dotenv

i_md_searches=md_searches()
i_mstr_global=mstr_global()
i_mstr_api=mstr_api()
i_redis_mstr_json=redis_mstr_json()
i_export_mstr_md=export_mstr_md()
i_read_gen=read_gen()
i_msic=msic()

env_file="..\\config\\streamlit.env"
load_dotenv(env_file)

with open('..\\config\\mstr_redis_y.yml', 'r') as openfile:
    mstr_redis_y = yaml.safe_load(openfile)

def grp_objtype_for_sml(obj_def_d_l):
    sml_mstr_obj_def_d_l={"dataset":{"obj_l":[],"ref_l":['https://github.com/semanticdatalayer/SML/blob/main/sml-reference/dataset.md']}, 
                          "dimension":{"obj_l":[],"ref_l":[]}}
    sml_mstr_type_d_l={"dataset":["logical_table"],
                        "dimension":["metric"]}

    for sml_md in sml_mstr_type_d_l:
        for ob in obj_def_d_l:
            
            if "subType" in ob.keys():
                if ob["subType"] in sml_mstr_type_d_l[sml_md]:
                    sml_mstr_obj_def_d_l[sml_md]["obj_l"].append(ob)
            
    return sml_mstr_obj_def_d_l

def fetch_multiple_sites(url_l):
    """
    Fetch content from multiple GitHub URLs.
    
    Args:
        url_list: List of GitHub URLs
        
    Returns:
        Dictionary with URL as key and content as value
    """
    results = {}
    
    for url in url_l:
        try:
            # Convert to raw URL
            raw_url = url.replace('github.com', 'raw.githubusercontent.com').replace('/blob/', '/')
            
            # Fetch content
            #response = requests.get(raw_url)
            content = requests.get(raw_url).text
            
            # Remove images
            content = re.sub(r'!\[.*?\]\([^)]*\)', '', content)
            content = re.sub(r'<img[^>]*/?>', '', content, flags=re.IGNORECASE)
            
            # Remove other non-text elements
            content = re.sub(r'<video[^>]*>.*?</video>', '', content, flags=re.DOTALL | re.IGNORECASE)
            content = re.sub(r'<svg[^>]*>.*?</svg>', '', content, flags=re.DOTALL | re.IGNORECASE)
            
            # Clean up whitespace
            results[url] = re.sub(r'\n{3,}', '\n\n', content)

            
            # Be nice to GitHub servers
            time.sleep(0.5)
            
        except Exception as e:
            print(f"Error fetching {url}: {e}")
            results[url] = None
    
    return results

def bld_user_msg(sml_file_type,obj_def_d_l):
    t_msg=f"Please generate sml file of the type {sml_file_type} based on the following MicroStrategy object definitions {obj_def_d_l}"
    return t_msg
    
def bld_sys_cont(all_results_ref):
    sys_cont="Your task is to tranform MicroStrategy metada into sml standard and generate a YAML file based on the following information and references."
    sys_cont=" you need to ensure, that you only generate valid SML Syntax"
    sys_cont=" to ensure valid syntax, please check or better test any generated syntax you need to ensure, that you only generate valid SML Syntax"
    sys_cont+=" Please ensure, that the YAML is well-formed and adheres to the SML schema and fits the AdScale requirements."
    sys_cont+= " Generate YAML output ONLY! Do not include any explanations, comments, or remarks.\nDo not add any text before or after the YAML. Output format: Pure YAML only."
    sys_cont+= f" Please ensure, that you stricly follow these instrunctions and samples {all_results_ref}"
    return sys_cont

def bld_ref_files(url_l):
    results=fetch_multiple_sites(url_l)
    all_results_ref = ""
    for url, content in results.items():
        if content:
            all_results_ref += content +"---"
    return all_results_ref  

def generate_sml_files(msg_t,sys_cont):

    temperature=1
    client = OpenAI(
        api_key=os.environ.get("PERPLEXITY_API_KEY"),
        # Set your API key in environment variables
        base_url="https://api.perplexity.ai"
    )
    try:
        # Create chat completion request
        messages = [
            {
                "role": "system",
                "content": sys_cont
            },
            {"role": "user", "content": msg_t}
        ]
        # print(messages)
        response = client.chat.completions.create(
            model="sonar-pro",  # Official model name for Perplexity-API
            messages=messages,
            temperature=temperature
        )
    except Exception as e:
        print(f"An error occurred: {e}")

    raw_msg=json.loads(response.json())["choices"][0]["message"]["content"]
    clean_yaml = re.sub(r'^```yaml\s*\n', '', raw_msg)
    clean_yaml = re.sub(r'\n```$', '', clean_yaml)
    return clean_yaml

def export_calculations_to_sml(obj_def_d_l, output_dir, ref_urls, sml_file_type, sub_types):
    """
    Export objects as SML calculation files.

    Args:
        obj_def_d_l: List of object definitions to process
        output_dir: Directory to save the generated SML files
        ref_urls: List of reference URLs for building context
        sml_file_type: The SML file type for the user message
        sub_types: List of subTypes to filter

    Returns:
        List of successfully exported file names
    """
    all_results_ref = bld_ref_files(ref_urls)
    sys_cont = bld_sys_cont(all_results_ref)
    exported_files = []

    for obj in obj_def_d_l:
        try:
            if obj.get("subType") in sub_types:
                name = obj["name"]
                t_msg = bld_user_msg(sml_file_type=sml_file_type, obj_def_d_l=obj)
                clean_yaml = generate_sml_files(msg_t=t_msg, sys_cont=sys_cont)

                file_path = os.path.join(output_dir, f'{name}.md')
                with open(file_path, 'w', encoding='utf-8') as f:
                    f.write(clean_yaml)
                exported_files.append(name)
        except Exception as e:
            print(f"Error processing object {obj.get('name', 'unknown')}: {e}")

    return exported_files

def bld_model_relations (fact_table_def): 
    relationships_d_l = []
    dimensions_l=[]
    for att in fact_table_def["attributes"]:
        rel_def_d={}
        rel_def_d["unique_name"]=fact_table_def["id"]
        rel_def_d["from"]={"dataset":fact_table_def["name"],"join_colums":["day_date"]}
        rel_def_d["to"]={"dimension":att["id"],"level":att["name"]}
        relationships_d_l.append(rel_def_d.copy())
        dimensions_l.append(att["id"])
    return {"relationships_d_l": relationships_d_l, "dimensions_l": dimensions_l}



################################
############recrusive Hier##########
def _collect_parents(attr_id, child_to_parents, visited=None):
    if visited is None:
        visited = set()
    if attr_id in visited:
        return []
    visited.add(attr_id)

    parents = child_to_parents.get(attr_id, [])
    all_ancestors = []
    for p in parents:
        all_ancestors.append(p)
        all_ancestors.extend(_collect_parents(p["objectId"], child_to_parents, visited))
    return all_ancestors

def get_all_parents_for_dims(dim_l, attribute_d_l):
    """
    For each attribute in dim_l, recursively traverses the relationship
    hierarchy in attribute_d_l and collects all parent (ancestor) attributes.

    Returns:
        dict keyed by dim id, each value is a list starting with the dim itself
        followed by ancestor dicts, each with keys: objectId, subType, name
    """
    # Build child_id -> list of parent info from all attribute relationships
    child_to_parents = {}
    for attr in attribute_d_l:
        for rel in attr.get("relationships", []):
            child_id = rel["child"]["objectId"]
            parent_info = rel["parent"]
            if child_id not in child_to_parents:
                child_to_parents[child_id] = []

            # Avoid duplicate parents
            if not any(p["objectId"] == parent_info["objectId"] for p in child_to_parents[child_id]):
                child_to_parents[child_id].append(parent_info)

    result = {}
    for dim in dim_l:
        ancestors = _collect_parents(dim["id"], child_to_parents)
        # Deduplicate while preserving order
        seen = set()
        unique_ancestors = []
        for a in ancestors:
            if a["objectId"] not in seen:
                seen.add(a["objectId"])
                unique_ancestors.append(a)
        dim_self = {"objectId": dim["id"], "subType": "attribute", "name": dim["name"]}
        result[dim["id"]] = [dim_self] + unique_ancestors

    return result

In [2]:
redis_con_d=mstr_redis_y["redis_env_d"]["redis_dev"]
project_prefix=mstr_redis_y["project_prefix"]
prefix_map=mstr_redis_y["prefix_map"]
searches_used_in_prp_d_l=mstr_redis_y["searches_used_in_prp_d_l"]


## Connect to MSTR & Redis

In [3]:
with open('..\\config\\user_d.json', 'r') as openfile:
    user_d = json.load(openfile)
conn_params =  user_d["conn_params"]
conn = Connection(**conn_params)
conn.headers['Content-type'] = "application/json"

i_redis_bi_analysis = redis_bi_analysis( 
    host=redis_con_d["host"],
    port=redis_con_d["port"],
    password=redis_con_d["password"],
    username=redis_con_d["username"],
    decode_responses=redis_con_d["decode_responses"]
)
conn.select_project("B7CA92F04B9FAE8D941C3E9B7E0CD754")

Connection to Strategy One Intelligence Server has been established.
No project selected.


## Fetch MSTR objects

### report tables

In [4]:
redis_env_p="mstr_dev"
redis_pre_obj="TABLE"
table_key_l=[]
table_d_l=[{"id":"24C30AD611D5AEC9C000E38A4CC5F24F","name":"lu_day"},
          {"id":"8D67933211D3E4981000E787EC6DE8A4","name":"lu_call_ctr"},
          {"id":"8D67933E11D3E4981000E787EC6DE8A4","name":"lu_category"},
          {"id":"8D67936811D3E4981000E787EC6DE8A4","name":"lu_employee"},
          {"id":"8D67937411D3E4981000E787EC6DE8A4","name":"lu_item"},
          {"id":"8D67938011D3E4981000E787EC6DE8A4","name":"lu_month"},
          {"id":"8D6793A411D3E4981000E787EC6DE8A4","name":"lu_quarter"},
          {"id":"8D6793C211D3E4981000E787EC6DE8A4","name":"lu_year"},
          {"id":"8D6793AA11D3E4981000E787EC6DE8A4","name":"lu_region"},
          {"id":"8D6793B611D3E4981000E787EC6DE8A4","name":"lu_subcateg"},
          {"id":"8D6793CE11D3E4981000E787EC6DE8A4","name":"order_detail"}]
for table in table_d_l:
    oby_key=f'{redis_env_p}:{redis_pre_obj}:{table["id"]}'
    table_key_l.append(oby_key)

### Fetch Dependencies for Specific Report

In [5]:
from mstr_robotics.redis_db import fetch_it_all

# Initialize the fetch_it_all class with your Redis connection
i_fetch_it_all = fetch_it_all(i_redis_bi_analysis)

# Define the root object key for the report definition you want to fetch
root_object_key = "mstr_dev:REPORT_DEFINITION:66FFEB6E49634F501160D2A4FB9BD78A"

# Fetch all dependencies recursively
all_dependencies = i_fetch_it_all.fetch_all_objects_recursively(
    root_object_l=[root_object_key],
    recursive_fg=True,
    batch_size=100
)

In [6]:
#prepare MD
direct_obj_check_d_l=[]
obj_type_set=set()
obj_to_export_d_l=[]
for obj in all_dependencies:
    try:
        if "information" in obj["definition"].keys():
            obj_type_set.add(obj["definition"]["information"]["subType"])
            if obj["definition"]["information"]["subType"] in ["report_grid","agg_metric"]:
                direct_obj_check_l=i_fetch_it_all.fetch_all_objects_recursively(root_object_l=[obj["obj_key"]]
                                             , recursive_fg=False)
                direct_obj_check_d_l.extend(direct_obj_check_l)
            if obj["definition"]["information"]["subType"] in ["report_grid","agg_metric"]:
                obj_to_export_d_l.extend([obj["obj_key"]])
        else:
            obj_type_set.add(obj["definition"]["subType"])
            if obj["definition"]["subType"] in ["metric","agg_metric"]:
                direct_obj_check_l=i_fetch_it_all.fetch_all_objects_recursively(root_object_l=[obj["obj_key"]]
                                             , recursive_fg=False)
                direct_obj_check_d_l.extend(direct_obj_check_l)
            if obj["definition"]["subType"] in ["metric","filter","prompt","role_transformation"]:
                obj_to_export_d_l.extend([obj["obj_key"]])       
    except:
        print("rreee")
        print(obj["obj_key"])

schema_obj_l=[]
for obj in direct_obj_check_d_l:
    try:
        if "information" in obj["definition"].keys():
            if obj["definition"]["information"]["subType"] in ["attribute","agg_metric"]:
                schema_obj_l.append(obj["obj_key"])
        
        else:
            #print(obj["definition"]["subType"])
            if obj["definition"]["subType"] in ["attribute","fact","hierarchy"]:
                schema_obj_l.append(obj["obj_key"])
    except:
        print("rreee")

schema_obj_l=i_msic.rem_dbl_in_l(schema_obj_l)
#schema_obj_d_l = remove_duplicates_by_obj_id(schema_obj_d_l)
obj_to_export_d_l.extend(schema_obj_l)
obj_to_export_d_l.extend(table_key_l)
obj_to_export_d_l
obj_id_l=[]

for obj in obj_to_export_d_l:
    obj_id_l.append(obj.split(":")[2])
obj_def_d_l=i_read_gen.get_proj_obj_def_by_id_l(conn,obj_id_l)
sml_mstr_obj_def_d_l=grp_objtype_for_sml(obj_def_d_l)



In [7]:
fact_table_l=["8D6793CE11D3E4981000E787EC6DE8A4"]
relationships_d_l=[]
dimensions_l=[]
for obj in obj_def_d_l:
    try:
        if obj["id"] in fact_table_l:
            model_rel_d=bld_model_relations(fact_table_def=obj)
            relationships_d_l.extend(model_rel_d["relationships_d_l"])
            dimensions_l.extend(model_rel_d["dimensions_l"])
    except:
        pass
    
model_d = {
    "unique_name": "TutorialModel",
    "object_type": "model",
    "label": "MicroStrategy Tutorial Model",
    "visible": True,
    "relationships": relationships_d_l,
    "dimensions": dimensions_l
}

with open(r'C:\coding\python_io\output_files\SML\models\model.md', 'w', encoding='utf-8') as f:
    yaml.dump(model_d, f, default_flow_style=False, allow_unicode=True, sort_keys=False)

In [ ]:
dataset_cols_l=["physicalTable", "isPartOfPartition","primaryDataSource", "secondaryDataSources","primaryLocale","name"]
mapped_schema_l=["attributes","facts","name"]
tableKey_l=["tableKey","name"]

sml_mstr_obj_def_d_l={"dataset":{"dataset_rag_l":[],"ref_l":['https://github.com/semanticdatalayer/SML/blob/main/sml-reference/dataset.md']} 
                      ,"mapped_schema_l":{"obj_l":[],"ref_l":[]}
                      ,"tablekeys_l":{"obj_l":[],"ref_l":[]}
                      ,"dimension":{"obj_l":[],"ref_l":[]}
                      }

sml_dataset_rag_l=[]
sml_dataset_expressions_rag_l=[]
sml_tablekeys_rag_l=[]
for obj in obj_def_d_l:
    if "physicalTable" in obj.keys():
        sml_mstr_obj_def_d_l["dataset"]["dataset_rag_l"].append(msic().select_dict_cols(obj, dataset_cols_l))
        sml_mstr_obj_def_d_l["mapped_schema_l"]["obj_l"].append(msic().select_dict_cols(obj, mapped_schema_l))
        sml_mstr_obj_def_d_l["tablekeys_l"]["obj_l"].append(msic().select_dict_cols(obj, tableKey_l))



In [ ]:
#Dataset defitions
all_results_ref = bld_ref_files(sml_mstr_obj_def_d_l["dataset"]["ref_l"])
sml_file_type="dataset.md"

t_msg = bld_user_msg(sml_file_type, obj_def_d_l=sml_mstr_obj_def_d_l["dataset"]["dataset_rag_l"])
sys_cont = bld_sys_cont(all_results_ref)
clean_yaml=generate_sml_files(msg_t=t_msg, sys_cont=sys_cont)

with open(rf'C:\coding\python_io\output_files\SML\dataset.md', 'w', encoding='utf-8') as f:
    f.write(clean_yaml)

In [8]:
# dimension / attribute extraction
attribute_d_l=[]
all_child_l=[]
all_parent_l=[]
dim_l=[]
child_all_parent={}

for obj in obj_def_d_l:
    try:
        if obj["subType"]=="attribute":
            attribute_d_l.append(obj)
    except:
        pass

for a in attribute_d_l:
    for r in a["relationships"]:
        all_parent_l.append(r["parent"]["objectId"])
        all_child_l.append(r["child"]["objectId"])
dimId_l=set(all_child_l)-set(all_parent_l)
dimId_l=list(dimId_l)

for a in attribute_d_l:
    if a["id"] in dimId_l:
        dim_l.append({"id":a["id"], "name":a["name"]})

dim_parents = get_all_parents_for_dims(dim_l, attribute_d_l)
for dim_name, parents in dim_parents.items():
    child_all_parent[dim_name] = {"level_attributes": parents, "secondary_attriutes": []}
child_all_parent

{'96ED3EC811D5B117C000E78A4CC5F24F': {'level_attributes': [{'objectId': '96ED3EC811D5B117C000E78A4CC5F24F',
    'subType': 'attribute',
    'name': 'Day'},
   {'objectId': '8D679D4411D3E4981000E787EC6DE8A4',
    'subType': 'attribute',
    'name': 'Month'},
   {'objectId': '8D679D4511D3E4981000E787EC6DE8A4',
    'subType': 'attribute',
    'name': 'Month of Year'},
   {'objectId': '8D679D4A11D3E4981000E787EC6DE8A4',
    'subType': 'attribute',
    'name': 'Quarter'},
   {'objectId': '8D679D5111D3E4981000E787EC6DE8A4',
    'subType': 'attribute',
    'name': 'Year'}],
  'secondary_attriutes': []},
 '8D679D4211D3E4981000E787EC6DE8A4': {'level_attributes': [{'objectId': '8D679D4211D3E4981000E787EC6DE8A4',
    'subType': 'attribute',
    'name': 'Item'},
   {'objectId': '8D679D3611D3E4981000E787EC6DE8A4',
    'subType': 'attribute',
    'name': 'Catalog'},
   {'objectId': '8D679D5011D3E4981000E787EC6DE8A4',
    'subType': 'attribute',
    'name': 'Supplier'},
   {'objectId': '54BABECD11D59

In [9]:
# Extract level attributes from logical tables
level_att_l = []
for obj in obj_def_d_l:
    if obj.get("subType") != "logical_table":
        continue
    
    table_id = obj["id"]
    table_name = obj["name"]
    
    for att in obj.get("attributes", []):
        for form in att.get("forms", []):
            lookup_id = form.get("lookupTable", {}).get("objectId")
            if lookup_id == table_id:
                level_att_d = {
                    "table_id": table_id,
                    "table_name": table_name,
                    "att_id": att["id"],
                    "att_name": att["name"]
                }
                level_att_l.append(level_att_d)

# Remove duplicates
level_att_l = i_msic.rem_dbl_dict_in_l(level_att_l)

# Collect all child attribute IDs from relationships
child_att_l = []
for la in level_att_l:
    for a in attribute_d_l:
        if la["att_id"] == a["id"]:
            for r in a["relationships"]:
                child_att_l.append(r["child"]["objectId"])

child_att_l = list(set(child_att_l))



In [10]:
sml_dimensions_d=child_all_parent.copy()
#print(sml_dimensions_d)
entry_att_hier_d={"96ED3EC811D5B117C000E78A4CC5F24F":"A00D4B5546C01F58A8691CB440BD8C41",
                  "8D679D4211D3E4981000E787EC6DE8A4":"FEA63419412BE1CA5ABA5B9632D6AB38",
                  "8D679D3F11D3E4981000E787EC6DE8A4":"9981A8614C30B583A641CCA84042F0A0"}
for dim in sml_dimensions_d.keys():
    user_hierarchy_def=user_hierarchies.get_user_hierarchy(connection=conn,project_id=conn.project_id,
                                        id=entry_att_hier_d[dim]).json()
    level_att_fin_l=[]
    for att in user_hierarchy_def["attributes"]:
        level_att_fin_l.append(att["objectId"])

    for att in sml_dimensions_d[dim]["level_attributes"]:
        #print(att["objectId"])
        if att["objectId"] not in level_att_fin_l:
            
            sml_dimensions_d[dim]["level_attributes"].remove(att)
            sml_dimensions_d[dim]["secondary_attriutes"].append(att)

with open('..\\config\\sml\\template_dim.yml', 'r') as openfile:
    dims_y = yaml.safe_load(openfile)

#pprint(dims_y)

for d in dims_y:
    #print(d)
    d_l=[]
    for dim in sml_dimensions_d:
        if d ==dim:
            #print(dim)
            for level_att in sml_dimensions_d[dim]["level_attributes"]:
                d_l.append(d)
                dims_y[d]["hierarchies"]["levels"].append({d:level_att["name"]})

dims_y

{'8D679D4211D3E4981000E787EC6DE8A4': {'unique_name': 'Product',
  'object_type': 'dimension',
  'label': 'Product Dimension',
  'type': 'standard',
  'hierarchies': {'unique_name': 'Product',
   'label': 'Product',
   'levels': [{'8D679D4211D3E4981000E787EC6DE8A4': 'Item'},
    {'8D679D4211D3E4981000E787EC6DE8A4': 'Supplier'},
    {'8D679D4211D3E4981000E787EC6DE8A4': 'Discontinued Code'},
    {'8D679D4211D3E4981000E787EC6DE8A4': 'Subcategory'},
    {'8D679D4211D3E4981000E787EC6DE8A4': 'Category'}]}},
 '96ED3EC811D5B117C000E78A4CC5F24F': {'unique_name': 'Time',
  'object_type': 'dimension',
  'label': 'Time dimension',
  'type': 'standard',
  'hierarchies': {'unique_name': 'Standard Time',
   'label': 'Time',
   'levels': [{'96ED3EC811D5B117C000E78A4CC5F24F': 'Day'},
    {'96ED3EC811D5B117C000E78A4CC5F24F': 'Month'},
    {'96ED3EC811D5B117C000E78A4CC5F24F': 'Quarter'},
    {'96ED3EC811D5B117C000E78A4CC5F24F': 'Year'}]}},
 '8D679D3F11D3E4981000E787EC6DE8A4': {'unique_name': 'Geography',


In [ ]:
# metrics
all_results_ref = bld_ref_files([
    "https://github.com/semanticdatalayer/SML/blob/main/sml-reference/metrics.md",
    "https://github.com/semanticdatalayer/sml-models-tutorials-adventureworks2012/tree/main/metrics"
    ])
sys_cont = bld_sys_cont(all_results_ref)
metric_d_l=[]
for obj in obj_def_d_l:
    try:
        if obj["subType"]=="metric":
            metric_d={}
            metric_d["conditionality"] = obj["conditionality"]
            metric_d["format"] = obj["format"]
            metric_d["name"] = obj["name"]
            metric_d["id"] = obj["id"]
            metric_d["object_type"] = obj["object_type"]
            metric_d["expression"] = obj["expression"]
            metric_d["expression_text"] = obj["expression"]["text"]
            metric_d_l.append(metric_d.copy())
            t_msg = bld_user_msg(sml_file_type="metric.md", obj_def_d_l=metric_d_l)

            clean_yaml=generate_sml_files(msg_t=t_msg, sys_cont=sys_cont)

            with open(rf'C:\coding\python_io\output_files\SML\metrics\{obj["name"]}.md', 'w', encoding='utf-8') as f:
                f.write(clean_yaml)
    except:
        pass


In [ ]:
dimensions_l

In [ ]:
#Extract model
all_results_ref = bld_ref_files(
    ["https://github.com/semanticdatalayer/SML/blob/main/sml-reference/model.md",
     #"https://github.com/semanticdatalayer/sml-models-tutorials-adventureworks2012/tree/main/models",
     ])

t_msg = bld_user_msg(sml_file_type="model.md", obj_def_d_l=attribute_d_l)
sys_cont = bld_sys_cont(all_results_ref)
sys_cont+=" In the user message you'll get a list with attribute definitions from MicroStrategy."
sys_cont+=f" You job is, to generate a correct model.md, according the SML definitions."
sys_cont+=f" Find detailed instructions here {all_results_ref}"
sys_cont+=f" when building the relationships please consider, that you find the rellation ships"
sys_cont +=" between lookuptables in the relationships section of each attribute"
sys_cont +="the relation of lu_tables and fact_table you can find in the attribute form expresions"
sys_cont+=f" Please ensure that you include the from_cardinality and to_cardinality in the output. Possible values are Many and One"
sys_cont+=f" be aware, when you define the relationships, that we are using a snowflake schema. This means that dimension tables are normalized into multiple related tables."
sys_cont+=f" Please ensure, that you check, that each entry attribute appears in ONLY one dimension. All parent attributes should be in the same dimension."
sys_cont+=f" the entry attribute are in this list {dim_l}. Each entry attribute, represents one dimension."
sys_cont+= " This means in other words, that each dimension is uniquely identified by one entry attribute. Further you need to ensure, that you give meanfull dimesion names"
sys_cont+= f" if you have more than {len(dim_l)}, something is  wrong. It's a good idea, to start with the dimensions"
clean_yaml=generate_sml_files(msg_t=t_msg, sys_cont=sys_cont)

clean_yaml=generate_sml_files(msg_t=t_msg, sys_cont=sys_cont)

with open(rf'C:\coding\python_io\output_files\SML\models\model.md', 'w', encoding='utf-8') as f:
    f.write(clean_yaml)

In [ ]:


# Example usage:
sml_export_conf_d={"output_dir":r'C:\coding\python_io\output_files\SML\calculation',
"ref_urls":[
    "https://github.com/semanticdatalayer/SML/blob/main/sml-reference/calculation.md",
    "https://github.com/semanticdatalayer/sml-models-tutorials-adventureworks2012/tree/main/calculations"
],
"sml_file_type":"calculation.md",
"sub_types":["filter", "role_transformation"]}

sml_export_conf_d={"output_dir":r'C:\coding\python_io\output_files\SML\metrics',
"ref_urls":[
    "https://github.com/semanticdatalayer/SML/blob/main/sml-reference/metrics.md",
    "https://github.com/semanticdatalayer/sml-models-tutorials-adventureworks2012/tree/main/metrics"
    ],
    "sml_file_type":"metric.md",
    "sub_types":["metric"]
}
exported = export_calculations_to_sml(
    obj_def_d_l=obj_def_d_l,
    output_dir=sml_export_conf_d["output_dir"],
    ref_urls=sml_export_conf_d["ref_urls"],
    sml_file_type=sml_export_conf_d["sml_file_type"],
    sub_types=sml_export_conf_d["sub_types"]
)

In [ ]:

for m in metric_d_l:
    print(m["name"])
    for t in m["expression"]["tokens"]:
        if "target" in t.keys():
            print(m["id"])
            print(t["target"]["objectId"])
            print(t["target"]["subType"])
            print(t["target"]["name"])
            #print(t["target"].keys())
for v in metric_d_l[6]["format"]["values"]:
    if v["type"]=="number_format":
        print (v["value"])


In [ ]:
#metrics
test_m_d=metric_d_l[0]
#test_m_d["SQL"]='"select	sum("a11"."tot_dollar_sales") AS "WJXBFS1" from	"public"."yr_category_sls"	"a11"'
test_m_d["SQL"]='''select	"a12"."month_id" AS "month_id",	max("a12"."month_desc") AS "month_desc0",	sum("a11"."tot_dollar_sales") AS "WJXBFS1" from	"public"."mnth_category_sls"	"a11"
	join	"public"."lu_month"	"a12"
	  on 	("a11"."month_id" = "a12"."prev_month_id")
group by	"a12"."month_id"'''
all_results_ref = bld_ref_files([
    "https://github.com/semanticdatalayer/SML/blob/main/sml-reference/metrics.md",
    "https://github.com/semanticdatalayer/sml-models-tutorials-adventureworks2012/tree/main/metrics"
    ])
sys_cont = bld_sys_cont(all_results_ref)

t_msg = bld_user_msg(sml_file_type="metric.md", obj_def_d_l=[test_m_d])
metric_y={"unique_name":test_m_d["id"]}
metric_y["object_type"]="metric"
metric_y["label"]=test_m_d["name"]
if "description" in test_m_d.keys():
    metric_y["description"]="metric"
if "format" in test_m_d.keys():
    for v in test_m_d["format"]["values"]:
        if v["type"]=="number_format":
            metric_y["format"]=v["value"]

metric_y["calculation_method"]=""
metric_y["dataset"]=""
metric_y["column"]=""
print(metric_y.keys())
sys_cont += f" in this request, you only need fill out the calculation_method, column and dataset fields from the dict {metric_y}"
sys_cont += "please exctract the avove fields value of the SQL field"
sys_cont += "consider, that the dataset is the table name in the FROM clause"
sys_cont += " please use only the column name without any alias or schema"
sys_cont += f" ensure, that the response yaml contains the fields fields {metric_y.keys()}, are included in the response"
clean_yaml=generate_sml_files(msg_t=t_msg, sys_cont=sys_cont)
clean_yaml

In [ ]:
#dimension
test_m_d=metric_d_l[0]
metric_text=""
for t in test_m_d["expression"]["tokens"]:
    #print(t)
    metric_text+=t["value"]
metric_text
test_m_d["SQL"]
#test_m_d["SQL"]='"select	sum("a11"."tot_dollar_sales") AS "WJXBFS1" from	"public"."yr_category_sls"	"a11"'
test_m_d["SQL"]='''select	"a12"."month_id" AS "month_id",	max("a12"."month_desc") AS "month_desc0",	sum("a11"."tot_dollar_sales") AS "WJXBFS1" from	"public"."mnth_category_sls"	"a11"
	join	"public"."lu_month"	"a12"
	  on 	("a11"."month_id" = "a12"."prev_month_id")
group by	"a12"."month_id"'''
all_results_ref = bld_ref_files([
    "https://github.com/semanticdatalayer/SML/blob/main/sml-reference/dimension.md",
    "https://learn.microsoft.com/en-us/sql/mdx/mdx-language-reference-mdx"
    ])
sys_cont = bld_sys_cont(all_results_ref)

t_msg = bld_user_msg(sml_file_type="metric.md", obj_def_d_l=[test_m_d])
metric_y={"unique_name":test_m_d["id"]}
metric_y["object_type"]="metric"
metric_y["label"]=test_m_d["name"]
if "description" in test_m_d.keys():
    metric_y["description"]="metric"
if "format" in test_m_d.keys():
    for v in test_m_d["format"]["values"]:
        if v["type"]=="number_format":
            metric_y["format"]=v["value"]

print(metric_y.keys())
sys_cont += f" in this request, you only need to tranform microstrategy metadata metric into a sml calculation"
sys_cont += f" you can find the logic in the SQL field {test_m_d['SQL']} and in the metric text_field {metric_text}"
sys_cont += f"  be aware, that this is a calculation and you must write valid MDX syntax"

sys_cont += f" ensure, that the response yaml contains the fields fields {metric_y.keys()}, are included in the response"
clean_yaml=generate_sml_files(msg_t=t_msg, sys_cont=sys_cont)
clean_yaml


In [ ]:
err_d_l=[]
for pre in project_prefix:
    env_prefix=project_prefix[pre]
    conn.select_project(pre)
    search_result = i_mstr_api.run_mstr_search(conn=conn,
                        search_id="96648F2B492150A6AA27DDB3744E32B4")
    print(search_result)

    if search_result["totalItems"] > 0:
        all_obj_d_l=search_result["result"]
        """
        err_d_l.append(i_redis_mstr_json.save_obj_json_to_redis(i_redis_bi_analysis=i_redis_bi_analysis                                                    
                                                        ,conn=conn
                                                        ,prefix_map=prefix_map
                                                        , all_obj_d_l=all_obj_d_l
                                                        , env_prefix=env_prefix)
                        )
        print(str(all_obj_d_l) + " uploaded to " + env_prefix)
        """
all_obj_d_l

In [ ]:

#RAG dimenisons
all_results_ref = bld_ref_files(["https://github.com/semanticdatalayer/SML/blob/main/sml-reference/dimension.md",
                                 "https://github.com/semanticdatalayer/sml-models-tutorials-adventureworks2012/tree/main/dimensions",
                                 "https://github.com/semanticdatalayer/sml-models-crisp-cpg-retail/tree/main/dimensions",])
for dim in dim_l:
    t_msg = bld_user_msg(sml_file_type="dimension.md", obj_def_d_l=attribute_d_l)
    sys_cont = bld_sys_cont(all_results_ref)
    sys_cont += f" you need to ensure, that you include all parent dependencies from the current Attribute {dim} "
    sys_cont += f" if you need to comment, please ensure, that it is clearly marked as comment. Best is to use '#' at the beginning of the line."
    clean_yaml=generate_sml_files(msg_t=t_msg, sys_cont=sys_cont)
    with open(rf'C:\coding\python_io\output_files\SML\dimensions\{dim["name"]}.md', 'w', encoding='utf-8') as f:
        f.write(clean_yaml)